In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom pathlib import Pathfrom statsmodels.graphics.tsaplots import plot_acf, plot_pacfsns.set_theme(style="whitegrid")data_path = Path("airpass.dat")airpass_series = pd.read_csv(data_path, header=None, names=["Passengers"], skiprows=1)airpass_series.index = pd.date_range(start="1949-01", periods=len(airpass_series), freq="MS")airpass_log = np.log(airpass_series["Passengers"])fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)axes[0].plot(airpass_series.index, airpass_series["Passengers"], color="#1f77b4")axes[0].set_title("Monthly International Airline Passengers")axes[0].set_ylabel("Passengers (thousands)")axes[1].plot(airpass_log.index, airpass_log, color="#ff7f0e")axes[1].set_title("Logarithm of Monthly Airline Passengers")axes[1].set_ylabel("Log Passengers")axes[1].set_xlabel("Year")fig.tight_layout()fig.savefig("airpass_original_vs_log.png", dpi=300, bbox_inches="tight")plt.show()print(    "Taking logarithms stabilizes the variance and converts the multiplicative seasonal pattern into an additive form, which is more suitable for ARIMA modeling.")

In [ ]:
log_diff = airpass_log.diff().dropna()fig, axes = plt.subplots(3, 1, figsize=(12, 12))axes[0].plot(log_diff.index, log_diff, color="#2ca02c")axes[0].set_title("First Difference of Log Passengers")axes[0].set_ylabel("Difference")plot_acf(log_diff, ax=axes[1], lags=48, zero=False)axes[1].set_title("ACF of First Difference")plot_pacf(log_diff, ax=axes[2], lags=48, method="ywadjusted", zero=False)axes[2].set_title("PACF of First Difference")for ax in axes:    ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.7)fig.tight_layout()fig.savefig("airpass_logdiff_diagnostics.png", dpi=300, bbox_inches="tight")plt.show()print(    "Seasonal differencing is still required because the time series plot and the ACF display strong residual seasonality with prominent spikes at multiples of 12 months.")

In [ ]:
log_diff_seasonal = log_diff.diff(12).dropna()fig, axes = plt.subplots(3, 1, figsize=(12, 12))axes[0].plot(log_diff_seasonal.index, log_diff_seasonal, color="#d62728")axes[0].set_title("Seasonal Difference (s=12) of First Difference")axes[0].set_ylabel("Difference")plot_acf(log_diff_seasonal, ax=axes[1], lags=48, zero=False)axes[1].set_title("ACF after Seasonal Differencing")plot_pacf(log_diff_seasonal, ax=axes[2], lags=48, method="ywadjusted", zero=False)axes[2].set_title("PACF after Seasonal Differencing")for ax in axes:    ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.7)fig.tight_layout()fig.savefig("airpass_seasonal_diagnostics.png", dpi=300, bbox_inches="tight")plt.show()print(    "Using a seasonal difference of 12 months is appropriate because the data are monthly and the differenced series now appears stationary with no dominant spikes at seasonal lags.")

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAXmodel = SARIMAX(    airpass_log,    order=(0, 1, 1),    seasonal_order=(0, 1, 1, 12),    enforce_stationarity=False,    enforce_invertibility=False,)model_fit = model.fit(disp=False)print(model_fit.summary())

In [ ]:
from statsmodels.graphics.gofplots import qqplotfrom statsmodels.stats.diagnostic import acorr_ljungboxresiduals = model_fit.resid.dropna()fig, axes = plt.subplots(2, 2, figsize=(12, 10))axes[0].plot(residuals.index, residuals, color="#9467bd")axes[0].set_title("Model Residuals")axes[0].set_ylabel("Residual")sns.histplot(residuals, bins=20, kde=True, color="#8c564b", ax=axes[1])axes[1].set_title("Residual Histogram")plot_acf(residuals, ax=axes[2], lags=48, zero=False)axes[2].set_title("Residual ACF")qqplot(residuals, line="s", ax=axes[3], markerfacecolor="#e377c2", markeredgecolor="black")axes[3].set_title("Residual Q-Q Plot")for ax in axes.flat:    ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.7)fig.tight_layout()fig.savefig("airpass_residual_diagnostics.png", dpi=300, bbox_inches="tight")plt.show()ljung_box = acorr_ljungbox(residuals, lags=[12, 24], return_df=True)print(ljung_box)

In [ ]:
forecast_steps = 24forecast_res = model_fit.get_forecast(steps=forecast_steps)forecast_mean_log = forecast_res.predicted_meanforecast_ci_log = forecast_res.conf_int()forecast_df = pd.DataFrame({    "Forecast": np.exp(forecast_mean_log),    "Lower 95%": np.exp(forecast_ci_log.iloc[:, 0]),    "Upper 95%": np.exp(forecast_ci_log.iloc[:, 1]),})forecast_df.index = pd.date_range(start=airpass_series.index[-1] + pd.offsets.MonthBegin(), periods=forecast_steps, freq="MS")fig, ax = plt.subplots(figsize=(12, 6))ax.plot(airpass_series.index, airpass_series["Passengers"], label="Observed", color="#1f77b4")ax.plot(forecast_df.index, forecast_df["Forecast"], label="Forecast", color="#ff7f0e")ax.fill_between(    forecast_df.index,    forecast_df["Lower 95%"],    forecast_df["Upper 95%"],    color="#ff7f0e",    alpha=0.2,    label="95% Prediction Interval",)ax.set_title("24-Month Forecast of Airline Passengers")ax.set_ylabel("Passengers (thousands)")ax.set_xlabel("Year")ax.legend()ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.7)fig.tight_layout()fig.savefig("airpass_forecast.png", dpi=300, bbox_inches="tight")plt.show()print(forecast_df.round(2))